# Features + walk-forward splits
Same per-stock features as the paper: **MACD, RSI, CCI, ADX** (+ close price), for the 30 DJIA members of each day. Plus 5 market features (VIX, turbulence, dispersion, stock-bond correlation, market return), kept for later.

All the feature math lives in `features.py` so `tests/test_lookahead.py` can check it. Run `1_data_collection.ipynb` first.

**Timing rule:** a row dated *t* only uses data up to the close of *t*. The agent sees it after the close and trades at the **t+1 close** (see `env.py`).

In [1]:
import pandas as pd
from features import build_features

prices = pd.read_parquet('data/prices.parquet')
membership = pd.read_parquet('data/membership.parquet')
vix = pd.read_parquet('data/vix.parquet')

# 1999 was only warm-up
asset, market = build_features(prices, membership, vix)
asset = asset[asset.index.get_level_values('date') >= '2000-01-01']
market = market[market.index >= '2000-01-01']
asset.to_parquet('data/asset_features.parquet')
market.to_parquet('data/market_features.parquet')
asset.shape, market.shape

((196166, 6), (6539, 5))

`asset` has one row per (date, permno) for every stock that's **in the DJIA** that day. That's the asset mask: no row means not tradable.

**For the env:** compute P&L from `ret` in `data/prices.parquet`, not from this table. On the day a stock delists (e.g. GM on 2009-06-02), CRSP adds one last row with the delisting return. That stock is no longer active, but whoever held it still takes that return.

In [2]:
asset.groupby('date').size().value_counts()   # stocks per day: 30 (29 for 4 days: GM delisted before the DJIA dropped it)

30    6535
29       4
Name: count, dtype: int64

In [3]:
asset.xs(14593, level='permno').tail()   # AAPL

,ticker,close,macd,rsi,cci,adx
date,,,,,,
2025-12-24,AAPL,273.81,0.282250,49.962268,-65.872420,21.970388
2025-12-26,AAPL,273.40,0.194262,49.119178,-58.052438,20.887851
2025-12-29,AAPL,273.76,0.151831,49.918302,-59.260284,20.030577
2025-12-30,AAPL,273.08,0.062611,48.372860,-59.726219,19.255849
2025-12-31,AAPL,271.86,-0.105326,45.642631,-68.692411,18.705154


In [4]:
market.describe().T

,count,mean,std,min,25%,50%,75%,max
vix,6539.0,19.838107,8.367654,9.140000,13.990000,17.780000,23.220000,82.690000
turbulence,6539.0,38.940530,46.582021,2.981654,17.915923,27.315866,43.394189,1415.625970
dispersion,6539.0,0.013280,0.008406,0.002988,0.008341,0.011031,0.015468,0.169709
stock_bond_corr,5834.0,-0.265561,0.298829,-0.836619,-0.513102,-0.298211,-0.031081,0.434860
mkt_ret_21,6539.0,0.006501,0.048339,-0.396511,-0.015260,0.013692,0.034485,0.224621


## Walk-forward splits
Train (expanding from 2000) → 3-month validation → 3-month test, rolled every quarter. Each interval includes its start date and excludes its end date.

In [5]:
FIRST_TEST = '2000-10-01'

test_start = pd.date_range(FIRST_TEST, market.index.max(), freq='QS')
folds = pd.DataFrame({
    'train_start': pd.Timestamp('2000-01-01'),
    'val_start': test_start - pd.DateOffset(months=3),
    'test_start': test_start,
    'test_end': test_start + pd.DateOffset(months=3),
})
folds.to_csv('data/folds.csv', index=False)
folds

,train_start,val_start,test_start,test_end
0,2000-01-01,2000-07-01,2000-10-01,2001-01-01
1,2000-01-01,2000-10-01,2001-01-01,2001-04-01
2,2000-01-01,2001-01-01,2001-04-01,2001-07-01
3,2000-01-01,2001-04-01,2001-07-01,2001-10-01
4,2000-01-01,2001-07-01,2001-10-01,2002-01-01
...,...,...,...,...
96,2000-01-01,2024-07-01,2024-10-01,2025-01-01
97,2000-01-01,2024-10-01,2025-01-01,2025-04-01
98,2000-01-01,2025-01-01,2025-04-01,2025-07-01
99,2000-01-01,2025-04-01,2025-07-01,2025-10-01
